# train.csv filtrado — Tienda 44, años 2016 y 2017

`train.csv` tiene ~125 millones de filas (5 GB), así que no lo cargamos completo: lo leemos por partes (*chunks*) y de cada parte nos quedamos solo con las filas de la tienda 44 y de los años 2016-2017. Al final el resultado es chico (una sola tienda), aunque tarde unos minutos en recorrer el archivo completo una vez.

**Antes de correr:** abre este archivo en VS Code, elige tu kernel de Python arriba a la derecha, y corre las celdas en orden (▷ o Shift+Enter).

In [ ]:
import pandas as pd
import numpy as np
import os

BASE_DIR = "C:/Tesis"


## Filtrado
Filtramos por año usando el texto de la fecha (`chunk["date"].str[:4]`) en vez de convertir toda la columna a fecha en cada chunk — es más rápido sobre 125 millones de filas. Recién al final, sobre el resultado ya chico, convertimos `date` a formato de fecha real.

In [ ]:
FILE_PATH = os.path.join(BASE_DIR, "train.csv")
CHUNK_SIZE = 1_000_000
STORE_NBR = 44          # en train.csv viene como número, no como texto
ANIOS = ["2016", "2017"]

partes = []

for i, chunk in enumerate(pd.read_csv(FILE_PATH, chunksize=CHUNK_SIZE)):
    filtro_anio = chunk["date"].str[:4].isin(ANIOS)
    filtro_tienda = chunk["store_nbr"] == STORE_NBR

    partes.append(chunk[filtro_anio & filtro_tienda])

    if (i + 1) % 20 == 0:
        print(f"Procesadas {(i + 1) * CHUNK_SIZE:,} filas leídas del archivo...")

train_tienda44 = pd.concat(partes, ignore_index=True)
train_tienda44["date"] = pd.to_datetime(train_tienda44["date"])

print(f"\nFilas obtenidas: {len(train_tienda44):,}")


## Revisión rápida del resultado

In [ ]:
train_tienda44.head(10)


In [ ]:
train_tienda44.info()


In [ ]:
print("Rango de fechas:", train_tienda44["date"].min(), "->", train_tienda44["date"].max())
print("Tienda:", train_tienda44["store_nbr"].unique())
print("Productos distintos (item_nbr):", train_tienda44["item_nbr"].nunique())


## Guardar el resultado
Como el filtrado tarda unos minutos, conviene guardar el resultado en un CSV chico para no tener que volver a leer los 125 millones de filas cada vez que quieras trabajar con estos datos — después solo cargas este archivo con `pd.read_csv("C:/Tesis/train_tienda44_2016_2017.csv")`.

In [ ]:
OUTPUT_PATH = os.path.join(BASE_DIR, "train_tienda44_2016_2017.csv")
train_tienda44.to_csv(OUTPUT_PATH, index=False)
print("Guardado en:", OUTPUT_PATH)
